Things need to do:
1. python -m venv [venv name]
2. source [venv name]/bin/activate
3. pip install ipykernel
4. ipython kernel install --user --name=[venv name]
5. Change kernal manually in local

In [ ]:
# !pip install polars openpyxl msoffcrypto-tool pandas xlsx2csv openpyxl fastexcel


In [ ]:
import polars as pl
from pathlib import Path
import pandas as pd
from IPython.display import display
from openpyxl import load_workbook
# from openpyxl.utils.exceptions import InvalidFileException
import msoffcrypto
from io import BytesIO
# import numpy as np
import gc
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas

In [ ]:
# Configure Polars display settings
pl.Config.set_tbl_cols(-1)  # Limit to 10 columns
pl.Config.set_tbl_rows(5)  # Limit to 20 rows
pl.Config.set_tbl_width_chars(200)  # Set the width of columns to 25 characters
pl.Config.set_fmt_str_lengths(200)
pl.Config.set_tbl_hide_column_data_types(True)
pl.Config.set_tbl_hide_dtype_separator(True)
pl.Config.set_tbl_hide_dataframe_shape(True)
pl.Config.set_tbl_cell_alignment("RIGHT")
pl.Config.float_precision=3

gc.collect()

In [ ]:

# # Path to the CSV file
# file_path = Path("OneDrive_1_10-14-2024/Enriched_Data/2885 Amplify ABM ESE One Time Enrichment_ESE_6570272 23 Sep 2024_1.xlsx").resolve()
# password = "FY25BA"

#  # Read the Excel file using pandas
# pandas_df = pd.read_excel(file_path,
#                         engine= 'openpyxl',
#                         sheet_name='Sheet1',  # Specify the sheet name
#                         # usecols=['col1', 'col2'],  # Read specific columns
#                         na_values=[None],  # Handle missing values
                        
#                         )

# # Convert the pandas DataFrame to a Polars DataFrame
# polars_df = pl.from_pandas(pandas_df)

# df = polars_df.with_columns([
#     pl.col(column).cast(pl.Utf8) for column in polars_df.columns
# ])
# # Display the DataFrame
# display(df)

In [ ]:
# Define the file path and the password
file_path = Path("OneDrive_1_10-14-2024/Enriched_Data/2885 Amplify ABM ESE One Time Enrichment_ESE_6570272 23 Sep 2024_1.xlsx").resolve()
password = "FY25BA"
sheet_nm = "data"

# Decrypt the file using msoffcrypto
decrypted = BytesIO()
with open(file_path, 'rb') as f:
    office_file = msoffcrypto.OfficeFile(f)
    office_file.load_key(password=password)
    office_file.decrypt(decrypted)

# Load the decrypted file into a pandas DataFrame
decrypted.seek(0)  # Reset the pointer to the beginning of the BytesIO object

polars_df = pl.read_excel(decrypted, sheet_name=sheet_nm, engine='calamine')

df = polars_df
# # Changing all columns data type to "String"
# df = polars_df.with_columns([
#     pl.col(column).cast(pl.Utf8) for column in polars_df.columns
# ])

df = df.rename(lambda col_nm: col_nm.lower())
df = df.rename(lambda col_nm: col_nm.replace(' ', '_'))

# display(df)

display(df.select(pl.col('id')).count())

In [ ]:
# for col in df.columns:
#     print(col,':', df[col].dtype)

Optimising dataframe

In [ ]:
# def reduce_memory_usage_pl(df, name):
#     """ Reduce memory usage by polars dataframe {df} with name {name} by changing its data types.
#         Original pandas version of this function: https://www.kaggle.com/code/arjanso/reducing-dataframe-memory-size-by-65 """
#     print(f"Memory usage of dataframe {name} is {round(df.estimated_size('mb'), 2)} MB")
#     Numeric_Int_types = [pl.Int8,pl.Int16,pl.Int32,pl.Int64]
#     Numeric_Float_types = [pl.Float32,pl.Float64]    
#     for col in df.columns:
#         col_type = df[col].dtype
#         try:
#             c_min = df[col].min()
#             c_max = df[col].max()
#         except Exception as e:
#             pass

#         if col_type in Numeric_Int_types:
#             if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
#                 df = df.with_columns(df[col].cast(pl.Int8))
#             elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
#                 df = df.with_columns(df[col].cast(pl.Int16))
#             elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
#                 df = df.with_columns(df[col].cast(pl.Int32))
#             elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
#                 df = df.with_columns(df[col].cast(pl.Int64))
#         elif col_type in Numeric_Float_types:
#             if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
#                 df = df.with_columns(df[col].cast(pl.Float32))
#             else:
#                 pass
#         # elif col_type == pl.Utf8:
#         #     df = df.with_columns(df[col].cast(pl.Categorical))
#         else:
#             pass
#     print(f"Memory usage of dataframe {name} became {round(df.estimated_size('mb'), 2)} MB")
#     return df


# # usage example
# df = reduce_memory_usage_pl(df, "polar_dataframe")

In [ ]:
# for col in df.columns:
#     print(col,':', df[col].dtype)

In [ ]:
filter_ba_contact_status = df.filter(pl.col('status') =="Electronically Enriched")\
# .filter( (pl.col('contact_country_ba') == 'United States') )
# .filter( (pl.col('contact_country_ba') == 'United States'))
    # .filter( (pl.col('contact_country_ba') == 'Germany') | (pl.col('contact_country_ba') == 'India') )
display(filter_ba_contact_status.select(pl.col('id').count()))

In [ ]:
filter_Change_in_Email = filter_ba_contact_status.filter(pl.col('adesk_contact_email_vs_contact_email_ba')=="Change in Email" ).unique()
display(filter_Change_in_Email.select(pl.col('id').count()))

In [ ]:
filter_email_domain_vs_url_null = filter_Change_in_Email.filter(pl.col('email_domain_vs_url').is_null()).unique()
display(filter_email_domain_vs_url_null.select(pl.col('id').count()))

In [ ]:
filter_Change_in_Email_null_but_same_email = filter_ba_contact_status.filter(pl.col('adesk_contact_email_vs_contact_email_ba').is_null() ).filter(pl.col('contact_email').str.to_lowercase() != pl.col('contact_email_ba').str.to_lowercase()).unique()
display(filter_Change_in_Email_null_but_same_email.select(pl.col('id').count()))

In [ ]:
# # need to work, don't know how to much extension of one string be part of another

# filter_email_domain_vs_url= filter_Change_in_Email.filter(pl.col('email_domain_vs_url')=='Different Domain')\
# .unique()
#     # .filter(pl.col('email_domain_cleaned').is_in(pl.col('url_cleaned')))\
#         # .unique()

# display(filter_email_domain_vs_url.select(pl.col('id').count()))


In [ ]:
new_contact_template = pl.concat([filter_email_domain_vs_url_null, filter_Change_in_Email_null_but_same_email], how='diagonal')
display(new_contact_template.select(pl.col('id').count()))

In [ ]:
# Fixing phone number
final_new_contact_template_df  = new_contact_template\
    .with_columns(pl.coalesce(pl.col(['contact_phone_ba','company_phone_ba'])).alias('contact_phone_ba'))
# Contact_Phone_BA, Contact_Mobile_BA, Company_Phone_BA
# Fixing product name 
final_new_contact_template_df = final_new_contact_template_df.with_columns(pl.coalesce(pl.col(['last_current_product_interest']), pl.lit('Dummy Product')).alias('last_current_product_interest'))

# Fixing contact country name 
final_new_contact_template_df = final_new_contact_template_df.with_columns(pl.coalesce(pl.col(['contact_country_ba', 'contact_country'])).alias('contact_country_ba'))

# Fixing email name 
final_new_contact_template_df = final_new_contact_template_df.with_columns(pl.coalesce(pl.col(['contact_email_ba', 'contact_email']).str.to_lowercase()).alias('Email') )

# Fixing city name 
final_new_contact_template_df = final_new_contact_template_df.with_columns(pl.coalesce(pl.col(['contact_city_ba', 'contact_city'])).alias('City') )

# Fixing Address name 
final_new_contact_template_df = final_new_contact_template_df.with_columns(
    pl.concat_str(
        [pl.col('contact_address1_ba'),pl.col('contact_address2_ba')], separator=' ', ignore_nulls=True
    ).alias('Address')
)


final_new_contact_template_df = final_new_contact_template_df.with_columns(
    pl.when(pl.col('contact_country_ba').str.contains('India')).then(pl.lit('IN'))
        .when(pl.col('contact_country_ba').str.contains('United States')).then(pl.lit('US'))
        .when(pl.col('contact_country_ba').str.contains('Germany')).then(pl.lit('DE'))
        .otherwise(pl.lit('No Match'))
        .alias('ISO_Code')
)

In [ ]:
final_new_contact_template_df = final_new_contact_template_df.select(
                                                                    pl.col('contact_first_name_ba').alias('First Name')  
                                                                    ,pl.col('contact_last_name_ba').alias('Last Name')
                                                                    ,pl.col('contact_job_title_ba').alias('Title')
                                                                    ,pl.col('Email')
                                                                    ,pl.col('contact_phone_ba').alias('Phone Number')
                                                                    ,pl.lit(None).alias('SendEmails')
                                                                    ,pl.lit(None).alias('MakePhoneCalls')
                                                                    ,pl.lit(None).alias('SendIndNews')
                                                                    ,pl.lit(None).alias('SendNewsLtr')
                                                                    ,pl.lit(None).alias('SendProdInfo')
                                                                    ,pl.lit(None).alias('SendTrailInfo')
                                                                    ,pl.lit('DataEnrichment').alias('SourceSystem')
                                                                    ,pl.col('account_sfdc_id').alias('AccountId')
                                                                    ,pl.col('account_csn').alias('Account CSN')
                                                                    ,pl.col('account_name').alias('Account Name')
                                                                )

final_new_contact_template_df = final_new_contact_template_df.unique()

display(final_new_contact_template_df.select(pl.all()) )

display(final_new_contact_template_df.select(pl.col('Email')).count() )

Export as csv file

In [ ]:
# final_new_contact_template_df.write_csv('final_new_contact_template.csv')

Export to Snowflake 

In [ ]:
# Convert Polars DataFrame to Pandas DataFrame
pandas_new_contact_df = final_new_contact_template_df.to_pandas()

# Snowflake connection details
account = 'tata.us-east-1'
user = ''
password = ''
database = 'DEV'
schema = 'DIBAKAR'
warehouse = 'BSM_QA_WAREHOUSE_MEDIUM'
role = 'BSM_ABMAMPLIFY_SECURED'

# Establish connection
ctx = snowflake.connector.connect(
            account=account,
            user=user,
            password=password,
            warehouse=warehouse,
            DATABASE=database,   
            SCHEMA=schema,      
            role=role)

table_name = 'new_contact'

# Truncate the table if it exists
truncate_query = f""" TRUNCATE TABLE IF EXISTS {database}.{schema}.{table_name} """

cursr = ctx.cursor()

try:
    # Execute the TRUNCATE TABLE command
    cursr.execute(truncate_query)
    print(f"Table {table_name} DROP successfully.")
except Exception as e:
    print(f"Error DROP table {table_name}: {e}")



# Write the DataFrame to Snowflake
success, nchunks, nrows, _ = write_pandas(conn=ctx, df=pandas_new_contact_df, table_name=table_name, database=database, schema=schema, overwrite=True, auto_create_table=True)

# Close the connection
ctx.close()

# Check if the upload was successful
if success:
    print(f"Successfully uploaded {nrows} rows in {nchunks} chunks.")
else:
    print("Failed to upload data.")